# v0.33.0 — Relation constraints

## 1. Intro

Since v0.33.0 an edge model (`BaseSurrealEdge`, v0.22.0) declares which tables its edges may join and whether both
ends must exist — SurrealDB's `TYPE RELATION IN … OUT … ENFORCED` — with the full SurrealDB-ORM's keys:

- `SurrealConfigDict(relation_in=…, relation_out=…, enforced=…, table_type=TableType.RELATION)` declares it on the
  model, checked when the class is defined;
- `relation_ddl()` renders the bare `DEFINE TABLE … TYPE RELATION` statement without a connection, and
  `await define_relation()` applies the constraint with one `DEFINE TABLE OVERWRITE` that keeps the table's other
  clauses (schema mode, `DROP`, comment, change feed, permissions);
- SurrealDB enforces it on every later write: a refused edge creation raises the ORM's `SurrealDbError`, or
  `SurrealDbNotFoundError` for a missing `ENFORCED` endpoint — the creation by table name, `relate("likes", …)`,
  included since v0.33.0.

`define_relation()` is the same on SurrealDB 2.x, 3.x and the embedded engine; only the rows of the README's 2.x/3.x
table differ: the wording of the server's refusals, the error of an endpoint both outside the ends and missing, and a
raw `ALTER TABLE` (§11). The notebook runs top to bottom on both lines: only the transaction (§9) and the line facts
(§11) print differently.

## 2. Connect

`SURREALDB_HOST` and `SURREALDB_PORT` choose the server (default `localhost:8000`), over WebSocket. The tables live in
their own database, `examples_relations`, reached through the named connection `rel_demo`. The notebook starts by
resetting that database, so it can be re-executed in place: `REMOVE DATABASE` and `DEFINE DATABASE` go in **one**
query, because on SurrealDB 3.x a bare `REMOVE DATABASE` leaves the session failing until the database is defined
again.

`apply_ddl()` runs SurrealQL through the SDK client's `query_raw()` and checks the status of **every** statement
(`client.query()` checks only the first one's); `table_definition()` reads a table's definition back from
`INFO FOR DB`.

In [1]:
import contextlib
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
SERVER_URL = f"ws://{HOST}:{PORT}/rpc"

SurrealDBConnectionManager.set_connection(  # the default connection
    url=SERVER_URL,
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
SurrealDBConnectionManager.add_connection(  # the database this notebook defines its relation tables in
    "rel_demo",
    url=SERVER_URL,
    user="root",
    password="root",
    namespace="examples",
    database="examples_relations",
)


async def apply_ddl(client, ddl: str) -> int:
    """Run SurrealQL statements and check every one of them; return how many ran."""
    response = await client.query_raw(ddl, {})
    if "error" in response:
        raise RuntimeError(response["error"])
    refused = [result["result"] for result in response["result"] if result["status"] != "OK"]
    if refused:
        raise RuntimeError(refused)
    return len(response["result"])


async def table_definition(table: str) -> str:
    """The definition of *table*, as INFO FOR DB reads it back."""
    return (await demo.query("INFO FOR DB;", {}))["tables"][table]


demo = await SurrealDBConnectionManager.get_client("rel_demo")
await apply_ddl(demo, "REMOVE DATABASE IF EXISTS examples_relations; DEFINE DATABASE examples_relations;")
print("Connections:", SurrealDBConnectionManager.list_connections())

Connections: ['default', 'rel_demo']


## 3. Models

`Person`, `Company`, `Post` and `Book` are ordinary node models. `Likes` is the README's edge model (section 34): a
person likes a post or a book, and both must exist (`enforced=True`). `Follows` joins two persons, without `ENFORCED`.
Every model runs on the connection `rel_demo`.

`relation_in` and `relation_out` take one table (`"person"`), several separated by `|` as SurrealQL writes them
(`"post | book"`), or a list or tuple of names; `None`, `""`, `[]` and `()` mean any table. Each name must be an
identifier: it is written in backquotes. `table_type=TableType.RELATION` is optional, since an edge model's table is
always `TYPE RELATION`. The keys are checked when the class is defined: below, a node model carrying `relation_in`
raises `TypeError` (derive it from `BaseSurrealEdge`).

In [2]:
from surrealdb import RecordID

from surreal_orm_lite import BaseSurrealEdge, BaseSurrealModel, SurrealConfigDict, TableType


class Person(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="person", connection="rel_demo")

    id: str | None = None
    name: str


class Company(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="company", connection="rel_demo")

    id: str | None = None
    name: str


class Post(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="post", connection="rel_demo")

    id: str | None = None
    title: str


class Book(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="book", connection="rel_demo")

    id: str | None = None
    title: str


class Likes(BaseSurrealEdge):
    model_config = SurrealConfigDict(
        table_name="likes",
        connection="rel_demo",
        table_type=TableType.RELATION,  # optional: an edge model's table is TYPE RELATION
        relation_in="person",
        relation_out=["post", "book"],
        enforced=True,
    )

    stars: int = 1


class Follows(BaseSurrealEdge):
    model_config = SurrealConfigDict(
        table_name="follows", connection="rel_demo", relation_in="person", relation_out="person"
    )


try:

    class Fan(BaseSurrealModel):  # a node model: relation constraints need an edge model
        model_config = SurrealConfigDict(table_name="fan", relation_in="person")

except TypeError as error:
    print(error)

Fan: relation constraints (relation_in) apply to an edge model only: derive Fan from BaseSurrealEdge.


## 4. Render and read the declaration

`relation_ddl()` renders the bare statement, without a connection: the names in backquotes, `IN … OUT … ENFORCED` in
that order, `DEFINE TABLE OVERWRITE` (or `IF NOT EXISTS` with `overwrite=False`). The getters read the declaration
back as tuples, the shape of introspection's `TableState` (§12). `get_table_type()` is `TableType.RELATION` for every
edge model and `TableType.NORMAL` for a node model without `table_type`.

In [3]:
print(Likes.relation_ddl())
print(Likes.relation_ddl(overwrite=False))
print(Follows.relation_ddl())
print()
print("in:", Likes.get_relation_in(), "| out:", Likes.get_relation_out(), "| enforced:", Likes.is_enforced())
print("Follows enforced:", Follows.is_enforced())
print("table types:", repr(Likes.get_table_type()), repr(Follows.get_table_type()), repr(Person.get_table_type()))

DEFINE TABLE OVERWRITE `likes` TYPE RELATION IN `person` OUT `post` | `book` ENFORCED;
DEFINE TABLE IF NOT EXISTS `likes` TYPE RELATION IN `person` OUT `post` | `book` ENFORCED;
DEFINE TABLE OVERWRITE `follows` TYPE RELATION IN `person` OUT `person`;

in: ('person',) | out: ('post', 'book') | enforced: True
Follows enforced: False
table types: <TableType.RELATION: 'relation'> <TableType.RELATION: 'relation'> <TableType.NORMAL: 'normal'>


## 5. Before the constraint

Nothing is enforced yet: the table `likes` does not exist, and the first edge written creates it as `TYPE ANY`, which
takes edges from and to any table. Here a company likes a post — an edge `Likes` will refuse once its constraint
applies (§8 comes back to it).

In [4]:
ada = await Person(id="ada", name="Ada").save()
bob = await Person(id="bob", name="Bob").save()
acme = await Company(id="acme", name="Acme").save()
graphs = await Post(id="graphs", title="Graphs").save()
sql = await Book(id="sql", title="SurrealQL").save()

early = await acme.relate(Likes, graphs)  # accepted: no constraint yet
print("written:", early.in_, "->", early.out)
print(await table_definition("likes"))

written: company:acme -> post:graphs
DEFINE TABLE likes TYPE ANY SCHEMALESS PERMISSIONS NONE


## 6. Apply the constraints

`follows` is defined by hand first, with a schema mode, a change feed and permissions, and a materialized view
(v0.32.0), `likes_per_post`, counts the likes of each post or book. Then `define_relation()` applies each model's
constraint and returns the statement it sent:

- `likes`, created by the edge of §5, holds only SurrealDB's defaults: the statement is `relation_ddl()`'s;
- `follows` keeps its clauses: `define_relation()` reads the stored definition (`INFO FOR DB`) and writes its other
  clauses back after the `TYPE RELATION` clause. Sent by hand, the bare `relation_ddl()` would reset them to
  SurrealDB's defaults: `SCHEMALESS`, no change feed, `PERMISSIONS NONE`, which hides every edge from record users.

The call is idempotent: run it at start-up.

In [5]:
await apply_ddl(
    demo,
    "DEFINE TABLE follows SCHEMAFULL CHANGEFEED 1d PERMISSIONS FOR select FULL; "
    "DEFINE TABLE likes_per_post AS SELECT count() AS n, out FROM likes GROUP BY out;",
)


async def like_counts() -> dict[str, int]:
    """The view likes_per_post: each post or book liked, with its count."""
    rows = await demo.query("SELECT n, out FROM likes_per_post;", {})
    return {str(row["out"]): row["n"] for row in sorted(rows, key=lambda row: str(row["out"]))}


print("follows, as written:", await table_definition("follows"))
print("likes_per_post:", await like_counts())
print()
for model in (Likes, Follows):
    print(await model.define_relation())
print()
for table in ("likes", "follows"):
    print(await table_definition(table))

follows, as written: DEFINE TABLE follows TYPE NORMAL SCHEMAFULL CHANGEFEED 1d PERMISSIONS FOR select FULL, FOR create, update, delete NONE
likes_per_post: {'post:graphs': 1}

DEFINE TABLE OVERWRITE `likes` TYPE RELATION IN `person` OUT `post` | `book` ENFORCED;
DEFINE TABLE OVERWRITE `follows` TYPE RELATION IN `person` OUT `person` SCHEMAFULL CHANGEFEED 1d PERMISSIONS FOR select FULL, FOR create, update, delete NONE;

DEFINE TABLE likes TYPE RELATION IN person OUT post | book ENFORCED SCHEMALESS PERMISSIONS NONE
DEFINE TABLE follows TYPE RELATION IN person OUT person SCHEMAFULL CHANGEFEED 1d PERMISSIONS FOR select FULL, FOR create, update, delete NONE


The table's records, fields, indexes, events and live queries, and the views that read it, are kept: `likes_per_post`
is still listed on `likes` (`INFO FOR TABLE`), still a view, and counts the edges written afterwards.

In [6]:
print("views on likes:", list((await demo.query("INFO FOR TABLE likes;", {}))["tables"]))
print(await table_definition("likes_per_post"))
print()
print("likes_per_post before:", await like_counts())
await ada.relate(Likes, graphs, stars=5)
print("likes_per_post after: ", await like_counts())

views on likes: ['likes_per_post']
DEFINE TABLE likes_per_post TYPE ANY SCHEMALESS AS SELECT count() AS n, out FROM likes GROUP BY out PERMISSIONS NONE

likes_per_post before: {'post:graphs': 1}
likes_per_post after:  {'post:graphs': 2}


Each `DEFINE TABLE` of a table with a change feed writes a `define_table` entry into it: the hand-written one, then
`define_relation()`'s. A raw `SHOW CHANGES` shows both; the ORM's `changes()` (v0.21.0) skips them and yields the
record changes only.

In [7]:
entries = await demo.query("SHOW CHANGES FOR TABLE follows SINCE 0 LIMIT 100;", {})
print("follows' change feed:", [kind for entry in entries for change in entry["changes"] for kind in change])

follows' change feed: ['define_table', 'define_table']


## 7. Enforcement

SurrealDB checks every edge created after `define_relation()`, at once and on every connection. Each creation path —
`relate()` with the edge model or by table name, an edge's `save()`, `objects().bulk_create()` — raises the ORM's
errors, with the SDK's error as the `__cause__`:

- an endpoint from a table outside `IN`/`OUT` raises `SurrealDbError`;
- with `ENFORCED`, an endpoint record that does not exist raises `SurrealDbNotFoundError`. It is a `SurrealDbError`,
  so `except SurrealDbError` handles every refused edge creation the same way;
- a batch with one refused edge writes nothing.

In [8]:
from surrealdb import ServerError

from surreal_orm_lite import SurrealDbError, SurrealDbNotFoundError

edge = await bob.relate(Likes, sql, stars=4)
print("accepted:", edge.in_, "->", edge.out, "| stars:", edge.stars)

try:
    await acme.relate(Likes, graphs)  # a company is not a person
except SurrealDbError as error:
    print("wrong table:", type(error).__name__)
    print("  the SDK's error is the cause:", isinstance(error.__cause__, ServerError))

try:
    await ada.relate(Likes, RecordID("book", "missing"))  # ENFORCED: the book must exist
except SurrealDbNotFoundError as error:
    print("missing book:", error)
    print("  a SurrealDbError too:", isinstance(error, SurrealDbError))

before = await Likes.objects().count()
try:
    batch = [Likes(in_=bob, out=graphs), Likes(in_=acme, out=sql)]  # the second edge is refused
    await Likes.objects().bulk_create(batch)
except SurrealDbError as error:
    print("bulk_create():", type(error).__name__)
print("edges before the batch:", before, "| after:", await Likes.objects().count())

try:
    await acme.relate("likes", sql)  # by table name: the ORM's error too, since v0.33.0
except SurrealDbError as error:
    print("by table name:", type(error).__name__)

accepted: person:bob -> book:sql | stars: 4
wrong table: SurrealDbError
  the SDK's error is the cause: True
missing book: There was a problem with the database: The record 'book:missing' does not exist
  a SurrealDbError too: True
bulk_create(): SurrealDbError
edges before the batch: 3 | after: 3
by table name: SurrealDbError


## 8. Records already in the table

SurrealDB checks the writes made after `define_relation()`, not the records already in the table: §5's edge, from a
company, stays and is still traversed. Two raw queries, through the model's `raw_query()`, find the edges to fix or
delete: those outside the ends (`record::tb(in) NOT IN [...]`) and the dangling ones (`in.id IS NONE OR out.id IS
NONE`: an endpoint record that no longer exists). Updating the nonconforming edge is refused; `delete()` always works.

In [9]:
OUTSIDE = (
    "SELECT VALUE id FROM likes WHERE record::tb(in) NOT IN ['person'] OR record::tb(out) NOT IN ['post', 'book']"
)
DANGLING = "SELECT VALUE id FROM likes WHERE in.id IS NONE OR out.id IS NONE"

outside = await Likes.raw_query(OUTSIDE)
print("outside the ends:", len(outside), "| dangling:", len(await Likes.raw_query(DANGLING)))
old = await Likes.objects().get(outside[0])
print("found:", old.in_, "->", old.out, "| the edge of §5:", old.id == early.id)

try:
    await old.merge(stars=3)
except ServerError as error:
    print("merge():", type(error).__name__)

await old.delete()
print("outside the ends after delete():", len(await Likes.raw_query(OUTSIDE)))

outside the ends: 1 | dangling: 0
found: company:acme -> post:graphs | the edge of §5: True
merge(): InternalError
outside the ends after delete(): 0


**A refused edge creation is an ORM error, a refused update the SDK's.** `merge()`, `update()`, `patch()`, the
`atomic_*()` helpers and `objects().bulk_update()` of a nonconforming edge raise the official SDK's `InternalError` (a
`surrealdb.ServerError`), as every update the server refuses in the lite: an event's `THROW`, a failed `patch()` test.
A dangling edge can still be updated: `ENFORCED` is checked when an edge is created.

## 9. Transactions

`define_relation(tx=tx)` reads the table's definition first, on committed state outside the block, then adds its
statement to the block. Over WebSocket on SurrealDB 3.x the transaction is **interactive**: the statement applies
inside the block, and an edge the new constraint refuses makes its `relate()` raise at once. Elsewhere (SurrealDB 2.x,
HTTP, the embedded engine) the transaction is **buffered**: everything is sent at the end of the block, whose exit
raises. Either way it is the transaction's `SurrealDbError` ("Transaction failed and rolled back: …") and nothing is
applied: the table keeps its type.

`Wrote` (a person wrote a book) gets its constraint in the block below. Its table already exists, `TYPE ANY`, created
by an edge written by table name.

In [10]:
async def native_tx_supported(client) -> bool:
    """True if the server exposes the SDK's native transaction RPC (SurrealDB 3.x)."""
    try:
        txn = await client.begin()
    except Exception:
        return False
    with contextlib.suppress(Exception):
        await client.cancel(txn)
    return True


class Wrote(BaseSurrealEdge):
    model_config = SurrealConfigDict(
        table_name="wrote", connection="rel_demo", relation_in="person", relation_out="book"
    )


await ada.relate("wrote", sql)  # creates the table, TYPE ANY
interactive = await native_tx_supported(demo)
print("Interactive transactions available (SurrealDB 3.x):", interactive)

try:
    async with SurrealDBConnectionManager.transaction("rel_demo") as tx:
        print("in the block:", await Wrote.define_relation(tx=tx))
        await acme.relate(Wrote, sql, tx=tx)  # a company: the constraint defined above refuses it
        print("buffered: the edge is queued, the block's exit raises")
except SurrealDbError as error:
    print(type(error).__name__ + ":", str(error).split(":")[0], "…")
print("wrote, after the block:", await table_definition("wrote"))

Interactive transactions available (SurrealDB 3.x): True
in the block: DEFINE TABLE OVERWRITE `wrote` TYPE RELATION IN `person` OUT `book`;
SurrealDbError: Transaction failed and rolled back …
wrote, after the block: DEFINE TABLE wrote TYPE ANY SCHEMALESS PERMISSIONS NONE


Because its read comes first, `define_relation()` does not see a definition of the same table queued earlier in the
same block (`tx.add("DEFINE TABLE wrote … SCHEMAFULL …")`): the statement it builds from the committed definition
replaces it at commit. Define the table's other clauses before the block.

## 10. The guards

`define_relation()` refuses, before any DDL, the cases where its statement would not do what the model asks:

- `overwrite=False` writes `DEFINE TABLE IF NOT EXISTS`, which leaves a table that exists as it is. On a table that is
  not a relation table — `wrote` is still `TYPE ANY` after the rolled-back block — the constraint would not apply:
  call `define_relation()` with `overwrite=True`, which keeps the table's other clauses.
- On a materialized view, `OVERWRITE` would drop the view's query and turn it into an ordinary table that no longer
  updates.

Both raise `SurrealDbError`, naming the table, and nothing is defined.

In [11]:
try:
    await Wrote.define_relation(overwrite=False)
except SurrealDbError as error:
    print(error)
print(await Wrote.define_relation())  # overwrite=True
print()

await apply_ddl(demo, "DEFINE TABLE stats AS SELECT count() AS n FROM person GROUP ALL;")


class Stats(BaseSurrealEdge):  # an edge model given a view's table by mistake
    model_config = SurrealConfigDict(table_name="stats", connection="rel_demo", relation_in="person")


try:
    await Stats.define_relation()
except SurrealDbError as error:
    print(error)
print("stats, still a view:", await table_definition("stats"))

Wrote.define_relation(overwrite=False): the table 'wrote' exists and is not a relation table; DEFINE TABLE IF NOT EXISTS would leave it as it is, and the relation constraint would not apply. Call define_relation() with overwrite=True, which keeps the table's other clauses. Nothing was defined.
DEFINE TABLE OVERWRITE `wrote` TYPE RELATION IN `person` OUT `book`;

Stats.define_relation(): the table 'stats' is a materialized view; defining it as a relation table would drop its query and turn it into an ordinary table that no longer updates. Give Stats another table_name, or remove the view first. Nothing was defined.
stats, still a view: DEFINE TABLE stats TYPE ANY SCHEMALESS AS SELECT count() AS n FROM person GROUP ALL PERMISSIONS NONE


## 11. What depends on the server line

`define_relation()` sends the same requests on every line: one `INFO FOR DB`, one `DEFINE TABLE OVERWRITE`, enforced
at once. Two things differ, both in what the server answers:

- The wording of its refusals: ``Found company:acme for field `in` … but expected a record<person>`` on SurrealDB 2.x
  and the embedded engine, ``Couldn't coerce value for field `in` …`` on 3.x.
- An endpoint both outside `IN`/`OUT` and missing, on an `ENFORCED` table: SurrealDB 2.x and the embedded engine
  report it missing (`SurrealDbNotFoundError`), 3.x reports the wrong table (`SurrealDbError`). `except
  SurrealDbError` catches both.

In [12]:
major = int((await demo.version()).rsplit("-", 1)[-1].split(".")[0])  # "surrealdb-3.3.4" -> 3
print(f"Server: SurrealDB {major}.x")

try:
    await ada.relate(Likes, RecordID("company", "ghost"))  # a company, and missing
except SurrealDbError as error:
    print("an endpoint outside the ends and missing:", type(error).__name__)

Server: SurrealDB 3.x
an endpoint outside the ends and missing: SurrealDbError


**Why `DEFINE TABLE OVERWRITE` and not `ALTER TABLE`**, which keeps the other clauses too: on SurrealDB 2.x and the
embedded engine, a raw `ALTER TABLE … TYPE RELATION …` of a table already written to is not enforced — writes keep the
old constraint while `INFO FOR DB` shows the new one — until a `DEFINE TABLE OVERWRITE` of the table; 3.x enforces it
at once. So the ORM never sends `ALTER TABLE`: `define_relation()` reads the other clauses and writes them back in one
`DEFINE TABLE OVERWRITE`, enforced at once on every line.

## 12. Read the constraint back

v0.30.0's `DatabaseIntrospector` reads the constraint back (`TableState.relation_in`, `relation_out`, `enforced`) in
the getters' shape, so a model compares with the database. `generate_models_from_db()` generates a `BaseSurrealEdge`
model carrying the keys, without `table_type=` (its base class says it). The file is written to a temporary
directory, whose path is not printed.

In [13]:
import pathlib
import tempfile

from surreal_orm_lite import DatabaseIntrospector, generate_models_from_db

schema = await DatabaseIntrospector("rel_demo", tables=["likes"]).introspect()
state = schema.tables["likes"]
print("table_type:", state.table_type, "| enforced:", state.enforced)
print("relation_in:", state.relation_in, "| relation_out:", state.relation_out)
print(
    "the model matches the database:",
    (Likes.get_relation_in(), Likes.get_relation_out(), Likes.is_enforced())
    == (state.relation_in, state.relation_out, state.enforced),
)
print("issues:", schema.issues)
print()

with tempfile.TemporaryDirectory() as directory:
    path = pathlib.Path(directory) / "relation_models.py"
    print(await generate_models_from_db(path, "rel_demo", tables=["likes"]))

table_type: relation | enforced: True
relation_in: ('person',) | relation_out: ('post', 'book')
the model matches the database: True
issues: ()

"""Models generated by surreal_orm_lite.generate_models_from_db() from a SurrealDB schema.

Review them before use. They describe tables that already exist: define_table()
keeps the clauses of the database they do not declare (a COMPUTED field stays
COMPUTED, a server DEFAULT stays), so it can be called on them; do not call
define_computed_fields() on them, it would replace the database's field definitions.
"""

from surreal_orm_lite import BaseSurrealEdge, SurrealConfigDict


class Likes(BaseSurrealEdge):
    # SCHEMALESS: fields not declared here are kept (extra="allow")
    model_config = SurrealConfigDict(
        table_name="likes",
        relation_in="person",
        relation_out=["post", "book"],
        enforced=True,
        connection="rel_demo",
        extra="allow",
    )



## 13. Removing the constraint

The model is the source of truth: an edge model without `relation_in`, `relation_out` and `enforced` (left out, `None`
or empty, and `enforced=False` alike) removes the constraint with `define_relation()` — here `LikesAny`, on the same
table. The table becomes `TYPE RELATION` with no ends, at once on every connection, and keeps its other clauses,
records, fields, indexes, events and views: the company's like is accepted now.

In [14]:
class LikesAny(BaseSurrealEdge):
    model_config = SurrealConfigDict(table_name="likes", connection="rel_demo")  # no relation key


print(await LikesAny.define_relation())
print(await table_definition("likes"))
edge = await acme.relate(LikesAny, graphs)
print("accepted now:", edge.in_, "->", edge.out)
print("likes_per_post:", await like_counts())

DEFINE TABLE OVERWRITE `likes` TYPE RELATION;
DEFINE TABLE likes TYPE RELATION SCHEMALESS PERMISSIONS NONE
accepted now: company:acme -> post:graphs
likes_per_post: {'book:sql': 1, 'post:graphs': 2}


`overwrite=False` removes nothing: `IF NOT EXISTS` leaves a relation table as it is. Turning the table back into
`TYPE ANY` is not offered, and it would not lift the constraint: SurrealDB keeps the `in` and `out` field definitions
(`TYPE record<person>`) through `ALTER TABLE … TYPE ANY` and `DEFINE TABLE OVERWRITE … TYPE ANY`, on every line. With
raw SurrealQL, `REMOVE FIELD in ON likes; REMOVE FIELD out ON likes;` drops them. `remove_relation()` deletes edges,
not the constraint.

## 14. Cleanup

Remove the `examples_relations` database, unregister `rel_demo` — `remove_connection()` closes its client — and close
every connection with `close_all_connections()`.

In [15]:
await apply_ddl(demo, "REMOVE DATABASE IF EXISTS examples_relations;")
await SurrealDBConnectionManager.remove_connection("rel_demo")
print("Connections:", SurrealDBConnectionManager.list_connections())
await SurrealDBConnectionManager.close_all_connections()
print("Done")

Connections: ['default']
Done
